# 03 · 모델 비교 + 임계값 튜닝

CV 재현 → PR 커브 → 운영 임계값 선택(불량 잡기 vs 오탐 trade-off).

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve
from src.data import load_secom
from src.features import build_preprocessor
from src.models import get_models
from src.evaluate import cross_val_report
X, y, ts = load_secom()

## CV 결과 재현 (run_baseline.py와 동일)

In [ ]:
rows = []
for name, clf in get_models(y).items():
    pipe = Pipeline([("pre", build_preprocessor()), ("clf", clf)])
    rows.append(cross_val_report(name, pipe, X, y))
pd.DataFrame(rows).sort_values("pr_auc", ascending=False)

## PR 커브 + 임계값 선택 (RandomForest)

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pipe = Pipeline([("pre", build_preprocessor()), ("clf", get_models(y)["rf"])]).fit(Xtr, ytr)
s = pipe.predict_proba(Xte)[:, 1]
p, r, t = precision_recall_curve(yte, s)
plt.plot(r, p, color="#163C73"); plt.xlabel("recall"); plt.ylabel("precision")
plt.title("PR curve — rf"); plt.grid(alpha=.3); plt.show()

In [ ]:
# 정밀도 >= 0.30 유지하며 최대 recall 주는 임계값
mask = p[:-1] >= 0.30
if mask.any():
    j = np.argmax(r[:-1] * mask)
    print(f"threshold={t[j]:.3f}  precision={p[j]:.3f}  recall={r[j]:.3f}")
else:
    print("정밀도 0.30 달성 불가 — 임계값/모델 재검토")

> 운영 관점: 불량 1개 놓침(미검출) 비용 ≫ 오탐 비용이면 **recall 우선**으로 임계값을 낮춘다.